# STEP-1

In [24]:
from pathlib import Path
import pandas as pd

DATA = Path.cwd().parent/"Dataset"/"LogChunks"

tables = []

for file in sorted((DATA / "build-failure-reason").rglob("*.xml")):
    # print(file)

    tables.append(pd.read_xml(file, parser="etree"))

df = pd.concat(tables, ignore_index=True)
df.columns = ["log", "keywords", "category", "chunk"]
df.head()

,log,keywords,category,chunk
0,C/git@git/failed/564416725.log,"not defined, -trace2_event_file,",0.0,+t/trace_schema_validator/trace_schema_validat...
1,C/git@git/failed/564815538.log,"not defined, -trace2_event_file,",0.0,+t/trace_schema_validator/trace_schema_validat...
2,C/git@git/failed/565119978.log,"not defined, -trace2_event_file,",0.0,t/trace_schema_validator/trace_schema_validato...
3,C/git@git/failed/565692415.log,"not defined, -trace2_event_file,",0.0,+t/trace_schema_validator/trace_schema_validat...
4,C/git@git/failed/565709337.log,"not defined, -trace2_event_file,",0.0,+t/trace_schema_validator/trace_schema_validat...


In [25]:
parts = df["log"].str.split("/", expand=True)
# print(parts)
df["language"] = parts[0]
df["repo"] = parts[1]
df["language"].nunique(), df["repo"].nunique()

(29, 80)

In [20]:
def count_lines(path):
    text = (DATA/"logs"/path).read_text(encoding="utf-8")
    return len(text.splitlines())

df["log_lines"] = df["log"].apply(count_lines)
df["chunk_lines"] = df["chunk"].apply(lambda chunk: len(chunk.splitlines()))
df[["log_lines", "chunk_lines"]].describe().round(0)

,log_lines,chunk_lines
count,797.0,797.0
mean,3718.0,57.0
std,5771.0,329.0
min,372.0,1.0
25%,988.0,3.0
50%,1598.0,7.0
75%,3986.0,24.0
max,39027.0,5929.0


In [21]:
df.head()

,log,keywords,category,chunk,language,repo,log_lines,chunk_lines
0,C/git@git/failed/564416725.log,"not defined, -trace2_event_file,",0.0,+t/trace_schema_validator/trace_schema_validat...,C,git@git,3320,2
1,C/git@git/failed/564815538.log,"not defined, -trace2_event_file,",0.0,+t/trace_schema_validator/trace_schema_validat...,C,git@git,3320,2
2,C/git@git/failed/565119978.log,"not defined, -trace2_event_file,",0.0,t/trace_schema_validator/trace_schema_validato...,C,git@git,3320,2
3,C/git@git/failed/565692415.log,"not defined, -trace2_event_file,",0.0,+t/trace_schema_validator/trace_schema_validat...,C,git@git,3318,2
4,C/git@git/failed/565709337.log,"not defined, -trace2_event_file,",0.0,+t/trace_schema_validator/trace_schema_validat...,C,git@git,3318,2


# STEP-2
### Need review

In [22]:
import re

def squash(text):
    return re.sub(r"[^A-Za-z0-9]", "", text)   # keep letters and digits only

def find_answer_lines(row):
    lines = (DATA / "logs" / row["log"]).read_text(encoding="utf-8").splitlines()
    squashed = [squash(line) for line in lines]
    chunk_text = squash(row["chunk"])
    pos = "".join(squashed).find(chunk_text)                 # where the answer starts, counted in characters
    ends = pd.Series([len(s) for s in squashed]).cumsum()    # characters used up by the end of each line
    start_line = (ends <= pos).sum() + 1
    end_line = (ends <= pos + len(chunk_text) - 1).sum() + 1
    return start_line, end_line

df[["start_line", "end_line"]] = df.apply(find_answer_lines, axis=1, result_type="expand")
row = df[df["log"] == "Python/nvbn@thefuck/failed/537373579.log"].iloc[0]
row["start_line"], row["end_line"]

(np.int64(1370), np.int64(1375))

In [27]:
# cell B: split by repo, then freeze
repos = pd.Series(df["repo"].unique()).sample(frac=1, random_state=42)   # shuffle the 80 repos
dev_repos = repos.head(20)

dev = df[df["repo"].isin(dev_repos)]                                                       # 20 repos to practise on
eval_set = df[~df["repo"].isin(dev_repos)].groupby("repo").sample(n=3, random_state=42)   # 3 logs from each of the other 60

OUT = Path.cwd().parent / "Dataset"
dev.to_json(OUT / "dev.jsonl", orient="records", lines=True)
eval_set.to_json(OUT / "eval.jsonl", orient="records", lines=True)
len(dev), len(eval_set)

(199, 180)

In [26]:
df.head()

,log,keywords,category,chunk,language,repo
0,C/git@git/failed/564416725.log,"not defined, -trace2_event_file,",0.0,+t/trace_schema_validator/trace_schema_validat...,C,git@git
1,C/git@git/failed/564815538.log,"not defined, -trace2_event_file,",0.0,+t/trace_schema_validator/trace_schema_validat...,C,git@git
2,C/git@git/failed/565119978.log,"not defined, -trace2_event_file,",0.0,t/trace_schema_validator/trace_schema_validato...,C,git@git
3,C/git@git/failed/565692415.log,"not defined, -trace2_event_file,",0.0,+t/trace_schema_validator/trace_schema_validat...,C,git@git
4,C/git@git/failed/565709337.log,"not defined, -trace2_event_file,",0.0,+t/trace_schema_validator/trace_schema_validat...,C,git@git
